In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
train_data = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
train_data.head()

In [ ]:
test_data = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")
test_data.head()

In [ ]:
women = train_data.loc[train_data.Sex == 'female']["Survived"]
women_rate = sum(women)/len(women)
print('% of women who survived', women_rate)

In [ ]:
men = train_data.loc[train_data.Sex == 'male']["Survived"]
men_rate = sum(men)/len(men)
print('% of men who survived', men_rate)

In [ ]:
from sklearn.model_selection import train_test_split

y = train_data.Survived

features = ["Pclass", "Sex", "SibSp", "Parch"]

X = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])

train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

def get_accuracy(n_estimators, max_depth, train_X, val_X, train_y, val_y):
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=1)
    model.fit(train_X, train_y)
    preds_val = model.predict(val_X)
    accuracy = accuracy_score(val_y, preds_val)
    return(accuracy)

In [ ]:
candidate_max_depth = [5, 10, 15, 20, 25, 50, 100, 250]
candidate_num_of_estimations = [5, 10, 15, 20, 25, 50, 100, 250]

best_max_depth = 5
best_estimation = 100
best_accuracy = 0

print("Initial Values:", best_estimation, best_max_depth, best_accuracy)

for max_depth in candidate_max_depth:
    for num_of_estimations in candidate_num_of_estimations:

        my_accuracy = get_accuracy(num_of_estimations, max_depth, train_X, val_X, train_y, val_y)
        print("Current Round", num_of_estimations, max_depth, my_accuracy)
    
        if (my_accuracy > best_accuracy):
            best_accuracy = my_accuracy
            best_max_depth = max_depth
            best_estimation = num_of_estimations
            print("Current Values:", best_estimation, best_max_depth, best_accuracy)

print("Best Choice Is:", best_estimation, best_max_depth, best_accuracy)

In [ ]:
# best parameters are 10, 10
model = RandomForestClassifier(n_estimators=10, max_depth=10, random_state=1)
model.fit(X, y)
predictions = model.predict(X_test)
print("Prediction Completed Successfully")

In [ ]:
output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)

print("Your submission was successfully saved!")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedStratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
import seaborn as sns

# 1. Paths & Load Data
target_name = 'Survived'
train_url = '/kaggle/input/competitions/titanic/train.csv'
test_url = '/kaggle/input/competitions/titanic/test.csv'

X_full = pd.read_csv(train_url)
X_test_full = pd.read_csv(test_url)

# Remove rows with missing target
y = X_full[target_name]
X_full.dropna(axis=0, subset=[target_name], inplace=True)
X_full.drop([target_name], axis=1, inplace=True)

X_full.head(10)

In [ ]:
# 2. Implement feature engineering and data cleaning 
for df in [X_full, X_test_full]:
    # 1. Total number of individuals
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

    # 2. Ticket price per person
    df['Fare_Per_Person'] = df['Fare'] / df['FamilySize']
    
    # 3. Logarithmic transformation
    df['LogFare_Per_Person'] = np.log1p(df['Fare_Per_Person'])

    # 4. Extract Deck
    df['Deck'] = df['Cabin'].astype(str).str[0]
    df['Deck'] = df['Deck'].replace({'n': 'U', 'T': 'U'})

    # 5. Extract Title FIRST (very important before Imputation)
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(['Mlle', 'Ms'], 'Miss')
    df['Title'] = df['Title'].replace('Mme', 'Mrs')    
    rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
    df['Title'] = df['Title'].replace(rare_titles, 'Rare')

    # 6. Impute missing Age based on Title median (after extracting Title)
    df['Age'] = df.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))

    # 7. Calculate Age*Class (after filling in the missing values in Age)
    df['Age*Class'] = df['Age'] * df['Pclass']

    # 8. Is Alone Feature
    df['Is_Alone'] = (df['FamilySize'] == 1).astype(int)


# Plot Fare
fig, axs = plt.subplots(1, 2, figsize=(12, 4), dpi=100)

# Original drawing of Fare (Skewed)
sns.kdeplot(X_full['Fare_Per_Person'], fill=True, ax=axs[0], color='red')
axs[0].set_title('Original Fare Per Person Distribution (Skewed)')
axs[0].set_xlabel('Fare')

# LogFare drawing after conversion (Normalized)
sns.kdeplot(X_full['LogFare_Per_Person'], fill=True, ax=axs[1], color='green')
axs[1].set_title('Log Transform: LogFare Per Person (Normalized)')
axs[1].set_xlabel('LogFare')

plt.tight_layout()
plt.show()

# cols_to_drop = ['SibSp', 'Parch', 'Fare', 'Fare_Per_Person', 'Age']
# X_full.drop(columns=cols_to_drop, inplace=True, errors='ignore')
# X_test_full.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [ ]:
# 3. Select Columns
categorical_cols = [cname for cname in X_full.columns 
                    if X_full[cname].nunique() < 10 and X_full[cname].dtype == "object"]

numerical_cols = [cname for cname in X_full.columns 
                  if X_full[cname].dtype in ['int64', 'float64']]

if 'PassengerId' in numerical_cols: 
    numerical_cols.remove('PassengerId')

my_cols = categorical_cols + numerical_cols

# Prepare Feature Sets
X_train_full, X_valid_full, y_train, y_valid = train_test_split(
    X_full[my_cols], y, train_size=0.8, test_size=0.2, random_state=0
)
X_test = X_test_full[my_cols].copy()
X_full_selected = X_full[my_cols].copy()

X_train_full.head(10)

In [ ]:
# 4. Pipeline Setup
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler())
])
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True)),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, numerical_cols),
    ('cat', cat_transformer, categorical_cols)
])

# Ensemble Modeling (Random Forest + XGBoost)
rf = RandomForestClassifier(random_state=42)
xgb = XGBClassifier(random_state=0, eval_metric='logloss')

ensemble_model = VotingClassifier(
    estimators=[('rf', rf), ('xgb', xgb)],
    voting='soft'
)

full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', ensemble_model)
])

print("Pipeline Created Successfully ")

In [ ]:
# 5. Calculating and graphing Mutual Information (MI)
print('--- Calculating & Plotting Mutual Information ---')

X_train_transformed = preprocessor.fit_transform(X_train_full)
feature_names = preprocessor.get_feature_names_out()

mi_scores = mutual_info_classif(X_train_transformed, y_train, random_state=0)
mi_series = pd.Series(mi_scores, index=feature_names).sort_values(ascending=True)

plt.figure(figsize=(10, round(mi_series.count() * 0.3)), dpi=100)
mi_series.plot(kind='barh', color='skyblue', edgecolor='black')
plt.title('Mutual Information Scores (Titanic Dataset)', fontsize=12, pad=15)
plt.xlabel('MI Score', fontsize=10)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

print("Top Feature Scores:")
print(mi_series.sort_values(ascending=False))


In [ ]:
# 6. Hyperparameter Tuning with GridSearchCV (خاصة بـ XGBoost)
param_grid = {
    'model__rf__n_estimators': [100, 200],
    'model__rf__max_depth': [4, 6, 8],
    'model__xgb__n_estimators': [50, 100, 150],
    'model__xgb__max_depth': [3, 4, 5],
    'model__xgb__learning_rate': [0.01, 0.03, 0.05],
    'model__xgb__subsample': [0.8, 1.0],
    'model__xgb__colsample_bytree': [0.8, 1.0]
}

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=42)

grid_search = GridSearchCV(
    estimator=full_pipeline, 
    param_grid=param_grid, 
    cv=cv, 
    scoring='accuracy', 
    n_jobs=-1, 
    verbose=2
)

print('Searching for best parameters for Voting Ensemble (RF + XGB)...')
grid_search.fit(X_train_full, y_train)

print('--- Result ---')
print('Best Parameters:', grid_search.best_params_)
print('Best Validation Accuracy:', grid_search.best_score_)

In [ ]:
# 7. Final Fit on Full Dataset with Best Parameters & Predict
best_pipeline = grid_search.best_estimator_
best_pipeline.fit(X_full_selected, y)

preds_test = best_pipeline.predict(X_test)

# 8. Submission File
output = pd.DataFrame({
    'PassengerId': X_test_full.PassengerId, 
    target_name: preds_test
})

output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")